In [1]:
# Relevant python functions
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import sys
import matplotlib.pyplot as plt
import folium
from collections import Counter
import matplotlib.patches as mpatches
import contextily as ctx
from matplotlib.ticker import ScalarFormatter, MaxNLocator
from shapely.geometry import box
import matplotlib.lines as mlines


# Import functions for inventory generation 
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
fxn_dir = os.path.join(parent_dir, "inventory_generation_functions")
sys.path.append(fxn_dir)
import functions_parcel_to_ftpt as inv_local 
import functions_general as fxns
import functions_inference as infr

In [2]:
# Set plotting CRS values for data manipulation and plotting
crs_main = '26910' # Used for data manipulation and storage
crs_plot = '4269' # Used for plotting 

# HAYWARD BOUNDS
xbounds = [-122.15, -122.02]
ybounds = [37.60, 37.69]


In [3]:
# Target Directory 
directory = './Inventory_Outputs/Synthesized_Local/'

# Make directory and intermediate directory
os.makedirs(directory, exist_ok=True)
dir_attribution = directory + 'FootprintAttribution/'
dir_intermediate = dir_attribution + 'Intermediate/'
os.makedirs(dir_intermediate, exist_ok=True)

# Figure Directory 
fig_dir = './Figures/General/'
os.makedirs(fig_dir, exist_ok=True)

## **Load Parcels**

In [4]:
# Load parcel geometry -- here, bring in extended 
parcels = fxns.json_to_gdf('./Input_Data/ProcessedData/Parcels_with_Extended_Data.json', crs_main)

## **Load Footprints and Tag with Possible APNs based on Geometry**

In [5]:
# Load building footprints 
footprints_original = fxns.json_to_gdf('./Input_Data/ProcessedFootprints/Hayward_Footprints.json', crs_main)

# Lower bound filters rows where the overlap is less than a given % of the footprint wiht a given parcel 
# Upper bound filters rows that have at least a given % of the footprint within a single parcel (drop other parcels associated with that footprint)
print(len(footprints_original))
footprints_filtered = inv_local.tag_ftpt_with_possible_apn(footprints_original, parcels, lower_bound=0, upper_bound=95)
print(len(footprints_filtered))

# Use these as footprints for merge
footprints = footprints_filtered.copy()

38355
50169


## **Update footprints to only include those with address data**

In [6]:
# Load address point data (attributed to footprints) 
points = fxns.json_to_gdf(dir_attribution + 'Address_Data_Attributed.json', crs_main)

# Filter for footprints that do/do not contain address points 
footprints_with_addresses = footprints[footprints['FootprintID'].isin(points['POINT_FootprintID'].to_list())]
footprints_no_addresses = footprints[~footprints['FootprintID'].isin(points['POINT_FootprintID'].to_list())]
print(len(footprints))
print(len(footprints_with_addresses))
print(len(footprints_no_addresses))


50169
44865
5304


In [ ]:
## Plot footprints with no addresses

### UNCOMMENT CODE TO PLOT INTERACTIVE MAP WITH FOOTPRINTS AND NSI POINTS
m = folium.Map(location=[parcels.copy().to_crs(crs_plot).geometry.iloc[0].centroid.y, parcels.copy().to_crs(crs_plot).geometry.iloc[0].centroid.x], zoom_start=12)

# Create a base map
# Add footprints (polygons)
folium.GeoJson(footprints_with_addresses.copy().to_crs(crs_plot), color = 'gray').add_to(m)
folium.GeoJson(footprints_no_addresses.copy().to_crs(crs_plot), color = 'red').add_to(m)
# folium.GeoJson(parcels[parcels['Use_Description_Hazus'].isin(['RES4','RES5','RES6'])].copy().to_crs(crs_plot), color = 'blue').add_to(m)

# display(m)


In [ ]:
# from IPython.display import display, HTML
# def folium_deepnote_show(m):
#     data = m.get_root().render()
#     data_fixed_height = data.replace('width: 100%;height: 100%', 'width: 100%').replace('height: 100.0%;', 'height: 609px;', 1)
#     display(HTML(data_fixed_height))

## **Attribute Parcel Data to Footprints**

In [ ]:
### MERGE ADDRESS POINTS WITH SCRAPED DATA ###

# Flag to determine if scaling is based on footprint area or building volumne 
use_height = True
new_ftpts_with_parcel = {}

counter = 0 
counter1 = 0 
unique_index = 0

# Record dropped parcels 
dropped_parcels = []

# Loop through parcels to populate dataframe 
print(len(parcels), 'parcels total (looping through these now)')
for index, row in parcels.iterrows():

    counter1 += 1
    if counter1 % 2000 == 0:
        print(counter1)


    # Get footprints within parcel 
    parcel_ftpt_address = footprints_with_addresses[footprints_with_addresses['APN_PQ']==row['APN_PQ']]
    parcel_ftpt_no_address = footprints_no_addresses[footprints_no_addresses['APN_PQ']==row['APN_PQ']]

    # ONE FOOTPRINT WITH ADDRESS DATA
    if len(parcel_ftpt_address) == 1: 

        # Associate all parcel data with given footprint 
        row['FootprintID'] = int(parcel_ftpt_address['FootprintID'].values[0])
        new_ftpts_with_parcel[unique_index] = row
        unique_index += 1
    



    # MULTIPLE FOOTPRINTS WITH ADDRESS DATA 
    elif len(parcel_ftpt_address) > 1:
            
        # Associate all parcel data with set of footprints 

        # Set all relevant footprint IDs to have parcel data associated with them
        for index, ftpt in parcel_ftpt_address.iterrows():
            cur_row = row.copy()
            cur_row['FootprintID'] = int(ftpt['FootprintID'])
            

            # Adjust scaling on various rows when splitting data across footprints
            if use_height: 
                num = ftpt['FootprintArea'] * ftpt['FootprintHeight'] # Volumne of each building
                denom = sum(parcel_ftpt_address['FootprintArea'] * parcel_ftpt_address['FootprintHeight']) # Summed volumne of everything in parcel
                factor = num / denom 
            else: 
                factor = ftpt['FootprintArea'] / sum(parcel_ftpt_address['FootprintArea'])

            # Assign scaling for area and value
            cur_row['Total_Value'] = cur_row['Total_Value'] * factor
            cur_row['Improvement_Value'] = cur_row['Improvement_Value'] * factor
            cur_row['Bldg_Area'] = cur_row['Bldg_Area'] * factor

            # Assign number of units - allow for 0 if original scrape has 0, but round to 1 in all other 0 cases 
            if cur_row['Num_Units'] > 0: 
                cur_row['Num_Units'] = max(np.round(cur_row['Num_Units'] * factor),1)

            # Assign number of buildings - allow for 0 if original scrape has 0, but round to 1 in all other 0 cases 
            if cur_row['Num_Bldg'] > 0: 
                cur_row['Num_Bldg'] = max(np.round(cur_row['Num_Bldg'] * factor),1)

            # Assign data to footprint 
            new_ftpts_with_parcel[unique_index] = cur_row.copy()
            unique_index += 1




    # ONE FOOTPRINT, BUT NO ADDRESS DATA 
    elif (len(parcel_ftpt_address) == 0) and len(parcel_ftpt_no_address) == 1 :

         # Associate all parcel data with given footprint 
        row['FootprintID'] = int(parcel_ftpt_no_address['FootprintID'].values[0])
        new_ftpts_with_parcel[unique_index] = row
        unique_index += 1





    # MULTIPLE FOOTPRINTS, BUT NO ADDRESS DATA 
    elif (len(parcel_ftpt_address) == 0) and len(parcel_ftpt_no_address) > 1 :

        # Associate all parcel data with set of footprints 

        # Set all relevant footprint IDs to have parcel data associated with them
        for index, ftpt in parcel_ftpt_no_address.iterrows():
            cur_row = row.copy()
            cur_row['FootprintID'] = int(ftpt['FootprintID'])

            # Adjust scaling on various rows when splitting data across footprints
            if use_height: 
                num = ftpt['FootprintArea'] * ftpt['FootprintHeight'] # Volumne of each building
                denom = sum(parcel_ftpt_no_address['FootprintArea'] * parcel_ftpt_no_address['FootprintHeight']) # Summed volumne of everything in parcel
                factor = num / denom 
            else: 
                factor = ftpt['FootprintArea'] / sum(parcel_ftpt_no_address['FootprintArea'])

            # Assign scaling for area and value
            cur_row['Total_Value'] = cur_row['Total_Value'] * factor
            cur_row['Improvement_Value'] = cur_row['Improvement_Value'] * factor
            cur_row['Bldg_Area'] = cur_row['Bldg_Area'] * factor

            # Assign number of units - allow for 0 if original scrape has 0, but round to 1 in all other 0 cases 
            if cur_row['Num_Units'] > 0: 
                cur_row['Num_Units'] = max(np.round(cur_row['Num_Units'] * factor),1)

            # Assign number of buildings - allow for 0 if original scrape has 0, but round to 1 in all other 0 cases 
            if cur_row['Num_Bldg'] > 0: 
                cur_row['Num_Bldg'] = max(np.round(cur_row['Num_Bldg'] * factor),1)

            # Assign data to footprint 
            new_ftpts_with_parcel[unique_index] = cur_row.copy()
            unique_index += 1




    # NO FOOTPRINTS
    elif (len(parcel_ftpt_address) == 0) and (len(parcel_ftpt_no_address) == 0):

        ## Find address points in parcel 
        points_with_apn = points[points['APN_PQ']==row['APN_PQ']]

        ## If there are points in the parcel, keep parcel data 
        if len(points_with_apn) > 0: 

            # Assign data witb missing footpirnt
            row['FootprintID'] = np.nan 
            new_ftpts_with_parcel[unique_index] = row
            unique_index += 1

        ## If there are no points in the parcel, drop parcel 
        else: 
            dropped_parcels.append(row['APN_PQ'])

    
    else:
        raise ValueError('Error: Code should not reach this point')


## Convert attributed parcel data to a geodataframe
parcels_attributed = pd.DataFrame.from_dict(new_ftpts_with_parcel, orient="index")

##### SAVE JSON FILE #####
fxns.gdf_to_json(parcels_attributed.copy(), dir_attribution + 'Parcel_Data_Attributed.json')


37282 parcels total (looping through these now)
2000
4000
6000
8000
10000
12000
14000
16000
18000
20000
22000
24000
26000
28000
30000
32000
34000
36000


## **Resolve cases with address/parcel, but no footprint**

In [ ]:
sdf

In [ ]:
parcels_attributed[parcels_attributed['FootprintID'].isna()]

,geometry,APN_PQ,Use_Description,Use_Code,Landslide,Liquefaction,Fault_Zone,Total_Value,Improvement_Value,Homeowner_Exemption,...,Bldg_Condition,Bldg_Quality,Bldg_Area,Num_Bldg,Num_Units,Num_Stories,Parking,Use_Description_Hazus,Construction_Hazus,FootprintID
179,POLYGON ((583318.0469696585 4165944.7918801033...,452-0084-097-00,"Vacant residential land, zoned 4 units or less",1000.0,No,No,No,47660.0,0.0,0.0,...,None,None,0.0,0.0,0.0,NaN,None,RES3_VAC,None,NaN
459,POLYGON ((579529.6991101083 4169261.6109825573...,431-0016-080-08,Exempt Public Agency,300.0,No,Yes,No,0.0,0.0,0.0,...,None,None,0.0,0.0,0.0,NaN,None,GOV1,None,NaN
475,POLYGON ((579907.3336466186 4169527.3543069037...,431-0008-050-00,Vacant commercial land (may include misc. imps),3000.0,No,No,No,184649.0,0.0,0.0,...,None,None,0.0,0.0,0.0,NaN,None,COM_VAC,None,NaN
476,POLYGON ((579920.4105805634 4169534.2192399283...,431-0008-049-00,Vacant commercial land (may include misc. imps),3000.0,No,No,No,184649.0,0.0,0.0,...,None,None,0.0,0.0,0.0,NaN,None,COM_VAC,None,NaN
1013,"POLYGON ((581771.965062632 4169580.122856146, ...",445-0304-016-00,SFR - Planned Development Tract with Common Area,1800.0,No,No,No,1004190.0,703290.0,0.0,...,B,80,2005.0,1.0,1.0,3.0,GARAGE,RES1_VAC,W,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45552,"POLYGON ((579576.3132017395 4168123.53559182, ...",443-0005-076-00,Exempt Public Agency,300.0,No,Yes,No,0.0,0.0,0.0,...,None,None,0.0,0.0,0.0,NaN,None,GOV1,None,NaN
45559,"POLYGON ((579260.1793480898 4169150.44762216, ...",431-0092-083-01,Exempt Public Agency,300.0,No,Yes,No,0.0,0.0,0.0,...,None,None,0.0,0.0,0.0,NaN,None,GOV1,None,NaN
45573,"POLYGON ((584752.875612861 4167247.6262758123,...",081D-2084-137-00,Exempt Public Agency,300.0,No,No,No,0.0,0.0,0.0,...,None,None,0.0,0.0,0.0,NaN,None,GOV1,None,NaN
45612,POLYGON ((580723.6565979061 4167187.6112074405...,443-0075-035-02,Exempt Public Agency,300.0,No,Yes,No,0.0,0.0,0.0,...,None,None,0.0,0.0,0.0,NaN,None,GOV1,None,NaN


In [ ]:
pd.DataFrame(new_ftpts_with_parcel)

,0,1,2,3,4,5,6,7,8,9,...,45051,45052,45053,45054,45055,45056,45057,45058,45059,45060
geometry,"POLYGON ((582781.4451387373 4166405.055676159,...",POLYGON ((582787.3237322584 4166396.1056780815...,POLYGON ((582801.3204874188 4166374.2264010063...,POLYGON ((582821.3537410746 4166343.1728938962...,"POLYGON ((582833.9170657531 4166323.617892127,...","POLYGON ((582853.4692794578 4166293.465812876,...",POLYGON ((582863.3311578389 4166278.0300896815...,"POLYGON ((582886.5572148812 4166241.574785441,...","POLYGON ((582899.444116093 4166221.304440353, ...",POLYGON ((582906.1226918367 4166210.9277522024...,...,POLYGON ((579704.7366393905 4166309.0499383705...,"POLYGON ((580272.8238970174 4170275.228349571,...","POLYGON ((582229.5893816368 4167727.85786494, ...","POLYGON ((580165.3573344926 4171285.927516895,...","POLYGON ((579367.621098172 4169142.6276879674,...","POLYGON ((582614.4791044849 4165522.676779401,...","POLYGON ((582614.4791044849 4165522.676779401,...","POLYGON ((582614.4791044849 4165522.676779401,...",POLYGON ((582477.3370207513 4165687.0177784776...,"POLYGON ((585854.4332886584 4163925.159942339,..."
APN_PQ,452-0068-093-00,452-0068-092-00,452-0068-091-00,452-0068-090-00,452-0068-089-02,452-0068-080-04,452-0068-079-00,452-0068-078-00,452-0068-077-01,452-0068-075-00,...,442-0071-139-01,428-0036-039-00,445-0230-025-00,415-0170-020-00,431-0106-029-00,465-0005-006-06,465-0005-006-06,465-0005-006-06,453-0095-029-00,078G-2926-001-01
Use_Description,Single family residential homes used as such,Single family residential homes used as such,Single family residential homes used as such,Single family residential homes used as such,Single family res home with non-economic 2nd unit,Single family residential homes used as such,Single family residential homes used as such,Single family residential homes used as such,Single family residential homes used as such,Single family residential homes used as such,...,Condominium Common Area or use,Automobile dealership,Commercial repair garage,"Vacant apartment land, capable of 5 or more units",Townhouse - Planned Development,Store/Office with Apts/Lofts,Store/Office with Apts/Lofts,Store/Office with Apts/Lofts,Single-tenant Retail Store,Exempt Public Agency
Use_Code,1100.0,1100.0,1100.0,1100.0,1200.0,1100.0,1100.0,1100.0,1100.0,1100.0,...,7390.0,8200.0,8100.0,7000.0,1500.0,3200.0,3200.0,3200.0,3100.0,300.0
Landslide,No,No,No,No,No,No,No,No,No,No,...,No,No,No,No,No,No,No,No,No,No
Liquefaction,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,...,Yes,No,No,Yes,Yes,Yes,Yes,Yes,Yes,No
Fault_Zone,No,No,No,No,No,No,No,No,No,No,...,No,No,No,No,No,No,No,No,No,No
Total_Value,318481.0,658856.0,128266.0,605228.0,733083.0,49723.0,775200.0,431816.0,176615.0,35472.0,...,0.0,440453.0,172597.0,238940.0,776360.0,351708.977496,478417.482953,396923.539551,903123.0,0.0
Improvement_Value,227837.0,416160.0,59503.0,428694.0,517078.0,25922.0,542640.0,226840.0,128531.0,21343.0,...,0.0,69692.0,19479.0,0.0,548352.0,111907.401931,152223.744576,126293.853494,271603.0,0.0
Homeowner_Exemption,7000.0,7000.0,7000.0,7000.0,5600.0,0.0,0.0,0.0,7000.0,7000.0,...,0.0,0.0,0.0,0.0,7000.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
sdf

In [ ]:
row

geometry                 POLYGON ((585854.4332886584 4163925.159942339,...
APN_PQ                                                    078G-2926-001-01
Use_Description                                       Exempt Public Agency
Use_Code                                                             300.0
Landslide                                                               No
Liquefaction                                                            No
Fault_Zone                                                              No
Total_Value                                                            0.0
Improvement_Value                                                      0.0
Homeowner_Exemption                                                    0.0
Year_Built                                                             NaN
Eff_Year_Built                                                         NaN
Bldg_Class                                                            None
Construction             

In [ ]:
parcel_ftpt_address

,geometry,FootprintArea,FootprintHeight,CensusBlock,CensusTract,FootprintID,APN_PQ


In [ ]:
693388.692373 + 301111.307627

994500.0

In [ ]:
len(new_ftpts_with_parcel)

45061

In [ ]:
new_ftpts_with_parcel[10]

geometry                 POLYGON ((582946.3165467593 4166149.77641286, ...
APN_PQ                                                     452-0080-020-00
Use_Description               Single family residential homes used as such
Use_Code                                                            1100.0
Landslide                                                               No
Liquefaction                                                            No
Fault_Zone                                                              No
Total_Value                                                       574482.0
Improvement_Value                                                 379270.0
Homeowner_Exemption                                                    0.0
Year_Built                                                          1961.0
Eff_Year_Built                                                      1962.0
Bldg_Class                                                            D55B
Construction             

In [ ]:
row

geometry                 POLYGON ((585854.4332886584 4163925.159942339,...
APN_PQ                                                    078G-2926-001-01
Use_Description                                       Exempt Public Agency
Use_Code                                                             300.0
Landslide                                                               No
Liquefaction                                                            No
Fault_Zone                                                              No
Total_Value                                                            0.0
Improvement_Value                                                      0.0
Homeowner_Exemption                                                    0.0
Year_Built                                                             NaN
Eff_Year_Built                                                         NaN
Bldg_Class                                                            None
Construction             

In [ ]:
ftpt

geometry           POLYGON Z ((582532.0875940638 4165564.36310536...
FootprintArea                                            1904.957785
FootprintHeight                                            12.657143
CensusBlock                                          060014382041005
CensusTract                                              06001438204
FootprintID                                                    36106
APN_PQ                                               465-0005-006-06
Name: 48230, dtype: object

In [ ]:
parcel_ftpt_address

,geometry,FootprintArea,FootprintHeight,CensusBlock,CensusTract,FootprintID,APN_PQ


In [ ]:
ftpt

geometry           POLYGON Z ((582532.0875940638 4165564.36310536...
FootprintArea                                            1904.957785
FootprintHeight                                            12.657143
CensusBlock                                          060014382041005
CensusTract                                              06001438204
FootprintID                                                    36106
APN_PQ                                               465-0005-006-06
Name: 48230, dtype: object

In [ ]:
new_ftpts_with_parcel[0]

geometry                 POLYGON ((582781.4451387373 4166405.055676159,...
APN_PQ                                                     452-0068-093-00
Use_Description               Single family residential homes used as such
Use_Code                                                            1100.0
Landslide                                                               No
Liquefaction                                                           Yes
Fault_Zone                                                              No
Total_Value                                                       318481.0
Improvement_Value                                                 227837.0
Homeowner_Exemption                                                 7000.0
Year_Built                                                          1948.0
Eff_Year_Built                                                      1948.0
Bldg_Class                                                            D50A
Construction             

In [ ]:
new_ftpts_with_parcel[1]

geometry                 POLYGON ((582787.3237322584 4166396.1056780815...
APN_PQ                                                     452-0068-092-00
Use_Description               Single family residential homes used as such
Use_Code                                                            1100.0
Landslide                                                               No
Liquefaction                                                           Yes
Fault_Zone                                                              No
Total_Value                                                       658856.0
Improvement_Value                                                 416160.0
Homeowner_Exemption                                                 7000.0
Year_Built                                                          1948.0
Eff_Year_Built                                                      1961.0
Bldg_Class                                                            D60B
Construction             

In [ ]:
row

geometry                 POLYGON ((585854.4332886584 4163925.159942339,...
APN_PQ                                                    078G-2926-001-01
Use_Description                                       Exempt Public Agency
Use_Code                                                             300.0
Landslide                                                               No
Liquefaction                                                            No
Fault_Zone                                                              No
Total_Value                                                            0.0
Improvement_Value                                                      0.0
Homeowner_Exemption                                                    0.0
Year_Built                                                             NaN
Eff_Year_Built                                                         NaN
Bldg_Class                                                            None
Construction             

In [ ]:
new_ftpts_with_parcel[0]

geometry                 POLYGON ((582781.4451387373 4166405.055676159,...
APN_PQ                                                     452-0068-093-00
Use_Description               Single family residential homes used as such
Use_Code                                                            1100.0
Landslide                                                               No
Liquefaction                                                           Yes
Fault_Zone                                                              No
Total_Value                                                       318481.0
Improvement_Value                                                 227837.0
Homeowner_Exemption                                                 7000.0
Year_Built                                                          1948.0
Eff_Year_Built                                                      1948.0
Bldg_Class                                                            D50A
Construction             

In [ ]:
new_ftpts_with_parcel[1]

geometry                 POLYGON ((582787.3237322584 4166396.1056780815...
APN_PQ                                                     452-0068-092-00
Use_Description               Single family residential homes used as such
Use_Code                                                            1100.0
Landslide                                                               No
Liquefaction                                                           Yes
Fault_Zone                                                              No
Total_Value                                                       658856.0
Improvement_Value                                                 416160.0
Homeowner_Exemption                                                 7000.0
Year_Built                                                          1948.0
Eff_Year_Built                                                      1961.0
Bldg_Class                                                            D60B
Construction             

In [ ]:
int(parcel_ftpt_address['FootprintID'].values[0])

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
parcel_ftpt_address

,geometry,FootprintArea,FootprintHeight,CensusBlock,CensusTract,FootprintID,APN_PQ
9193,"POLYGON Z ((582760.451 4166401.215 0, 582762.0...",2012.196244,12.268293,060014379002008,06001437900,10808,452-0068-093-00


In [ ]:
new_ftpts_with_parcel.keys()

dict_keys([np.int64(10808)])

In [ ]:
row['Use_Description_Hazus']

'RES1'

In [ ]:
parcel_ftpt_no_address

,geometry,FootprintArea,FootprintHeight,CensusBlock,CensusTract,FootprintID,APN_PQ


In [ ]:
### MERGE ADDRESS POINTS WITH SCRAPED DATA ###

# Flag to determine if scaling is based on footprint area or building volumne 
use_height = True

# Loop through scraped data to populate dataframe 
print(len(scraped_merge), 'parcels total (looping through these now)')
for index, row in scraped_merge.iterrows():

    # Find the corresponding rows of addresses within the current parcel 
    gdf_index = address_merge[address_merge['APN_PQ'] == row['APN_PQ']].index
    group = address_merge.loc[gdf_index] 

    # Get footprints within parcel 
    parcel_ftpt = footprints_filtered[footprints_filtered['APN_PQ']==row['APN_PQ']]


    ### START RULESETS FOR MERGE ###


    # MANUSCRIPT CASE 5: NO FOOTPRINTS, ONE OR MORE ADDRESS POINT(S)
    if len(parcel_ftpt) == 0: 

         # One parcel and 1 address point, associate parcel and address point 
        if len(gdf_index) == 1:
            address_merge = inv_local.associate_scraped_data(address_merge.copy(), gdf_index, row, scraped_merge.columns)
            address_merge.loc[gdf_index, 'NoScrapedData'] = 'No Footprints Available in Parcel'

        # One parcel and several address points, split data between addresses 
        else: 
            factor = 1/len(gdf_index)
            address_merge = inv_local.associate_scraped_data(address_merge, gdf_index, row, scraped_merge.columns)
            address_merge.loc[gdf_index, 'Scrape_Total_Value_Update'] = row['Total_Value'] * factor
            address_merge.loc[gdf_index, 'Scrape_Improvement_Value_Update'] = row['Improvement_Value'] * factor
            address_merge.loc[gdf_index, 'Scrape_Bldg_Area_Update'] = row['Bldg_Area'] * factor
            address_merge.loc[gdf_index, 'Scrape_Num_Units_Update'] = (np.round(max(row['Num_Units'] * factor, 1)))
            address_merge.loc[gdf_index, 'Scrape_Num_Bldg_Update'] = (np.round(max(row['Num_Bldg'] * factor, 1)))
            address_merge.loc[gdf_index, 'DataUpdate'] = 'Split Parcel Data Between Addresses - No Footprints Available in Parcel'
            address_merge.loc[gdf_index, 'Num_Addresses_Split'] = len(gdf_index)
            address_merge.loc[gdf_index, 'NoScrapedData'] = 'No Footprints Available in Parcel'


    
    # There are footprints available in the parcel
    else: 

        # ONE PARCEL, ONE ADDRESS POINT 
        if len(gdf_index) == 1:

            # Associate parcel and address point 
            address_merge = inv_local.associate_scraped_data(address_merge.copy(), gdf_index, row, scraped_merge.columns)
        
            # MANUSCRIPT CASE 1: ONE FOOTPRINT, ONE ADDRESS POINT: Associate address point and parcel data to footprint, even if not intersecting 
            if len(parcel_ftpt) == 1:
                address_merge.loc[gdf_index, 'FootprintID'] = parcel_ftpt['FootprintID'].values[0]

            # MANUSCRIPT CASE 2: MULTIPLE FOOTPRINTS, ONE ADDRESS POINT: Associate address point and parcel data to closest footprint 
            elif (len(parcel_ftpt) > 1):
                
                # Spatial join address and footprints 
                address_with_ftpt = gpd.sjoin(group, parcel_ftpt[['FootprintID','geometry']], how="left", predicate='within')

                # If address not directly within a footprint, locate closest footprint
                if np.isnan(address_with_ftpt['FootprintID_right'].values[0]): 
                    address_with_ftpt = inv_local.find_nearest(address_with_ftpt, parcel_ftpt)
                    address_merge.loc[gdf_index, 'FootprintID'] = address_with_ftpt['ClosestFtpt_ID'].values[0]
                        
                # If address is directly wihtin a footprint, associate address with footprint  
                else: 
                    address_merge.loc[gdf_index, 'FootprintID'] = address_with_ftpt['FootprintID_right'].values[0]



        # Multiple Address Points associated with Parcel 
        elif len(gdf_index) > 1:

            # Merge and associate footprints 
            if use_height: 
                address_with_ftpt = gpd.sjoin(group, parcel_ftpt[['FootprintID','FootprintArea','FootprintHeight','geometry']], how="left", predicate='within')
            else: 
                address_with_ftpt = gpd.sjoin(group, parcel_ftpt[['FootprintID','FootprintArea','geometry']], how="left", predicate='within')
            address_merge.loc[gdf_index, 'FootprintID'] = address_with_ftpt['FootprintID_right'].values

            # Check if there are accessory dwelling units present in the parcel 
            adu_index = group[group['FeatureCode'] == 'Single-Family Dwelling Accessory Suite'].index




            # If no ADUs present in the parcel 
            if len(adu_index) == 0: 

                # MANUSCRIPT CASE 3: 1 OR MORE FOOTPRINTS, MULTIPLE POINTS, ALL POINTS WITHIN FOOTPRINT(S)
                # If all points are within footprints, associate parcel information with one point per footprint for later use, making notes for all other points 
                if address_with_ftpt.loc[gdf_index, 'FootprintID_right'].notna().all():
                    address_merge, drop_idx = inv_local.associate_scraped_split_multiple_addresses(address_merge, gdf_index, row, scraped_merge.columns, address_with_ftpt, 'FootprintID_right', use_height)
                
                # MANUSCRIPT CASE 4: 1 OR MORE FOOTPRINTS, MULTIPLE POINTS, SOME OR ALL POINTS NOT INTERSECTING WITH FOOTPRINT(S)
                # If some or all points are not located within a footprint
                else:

                    # Find the nearest footprint for each non-ADU address
                    address_with_ftpt = inv_local.find_nearest(address_with_ftpt, parcel_ftpt)
                    address_with_ftpt_close_idx = address_with_ftpt[address_with_ftpt['DistanceToFtpt'] <= 10].index
                    address_with_ftpt_far_idx = address_with_ftpt[address_with_ftpt['DistanceToFtpt'] > 10].index

                    # If there are no points close to a building footprint, associate data with closest footprint regardless
                    if len(address_with_ftpt_close_idx) == 0: 
                
                            # Reset footprints and building area for cases where data is close and map appropriate information to address_with_ftpt
                            address_merge.loc[gdf_index, 'FootprintID'] = address_with_ftpt.loc[gdf_index, 'ClosestFtpt_ID'].values
                            address_with_ftpt = inv_local.map_closest_footprints(address_with_ftpt.copy(), footprints_orig, use_height)

                            # Associate information with one point per footprint, making notes for all other points
                            address_merge.loc[gdf_index, 'NoScrapedData'] = 'Has Scraped Data, but 10m Away - No Close Points'   
                            address_merge, drop_idx = inv_local.associate_scraped_split_multiple_addresses(address_merge, gdf_index, row, scraped_merge.columns, address_with_ftpt, 'ClosestFtpt_ID', use_height)

                    
                    # If there are some points close to building footprints, associate addresses to closest footprints, associate parcel with all footprints containing data 
                    else:    

                        # Reset footprints and building area for cases where data is close and map appropriate information to address_with_ftpt
                        address_merge.loc[address_with_ftpt_close_idx, 'FootprintID'] = address_with_ftpt.loc[address_with_ftpt_close_idx, 'ClosestFtpt_ID'].values
                        address_with_ftpt = inv_local.map_closest_footprints(address_with_ftpt.copy(), footprints_orig, use_height)
                     
                        # Associate information with one point per footprint, making notes for all other points 
                        address_merge, drop_idx = inv_local.associate_scraped_split_multiple_addresses(address_merge, address_with_ftpt_close_idx, row, scraped_merge.columns, address_with_ftpt, 'ClosestFtpt_ID', use_height)
                        address_merge.loc[address_with_ftpt_far_idx, 'NoScrapedData'] = 'Point Not Within 10m of Footprint'  
                

            # REPEAT LOGIC, BUT WITH LOGIC TO HANDLE ADUs IN DATA
            else:

                # Find non-ADU points
                non_adu_indices = group[group['FeatureCode'] != 'Single-Family Dwelling Accessory Suite'].index

                # If parcel has ADU(s) and only one non-ADU point, associate data with non-ADU address. This is the same as the one address, one footprint method above (MANUSCRIPT CASE 1)
                if len(non_adu_indices) == 1: 
                    
                    # Associate parcel and non-ADU address point, make notes for ADU points
                    address_merge = inv_local.associate_scraped_data(address_merge.copy(), non_adu_indices, row, scraped_merge.columns)
                    address_merge.loc[adu_index, 'NoScrapedData'] = 'Scraped Data Associated with Non-ADU Structure'  

                    # If non-ADU unit does not have a footprint, update with closest footprint
                    if np.isnan(address_with_ftpt.loc[non_adu_indices, 'FootprintID_right'].values[0]): 
                        address_with_ftpt = inv_local.find_nearest(address_with_ftpt, parcel_ftpt)
                        address_merge.loc[non_adu_indices, 'FootprintID'] = address_with_ftpt.loc[non_adu_indices, 'ClosestFtpt_ID'].values[0]
                    
                # If parcel has ADU(s) and more than one non-ADU point
                elif len(non_adu_indices) > 1: 

                    # If parcel contains ADU(s), and all non-ADU points are within a footprint (MANUSCRIPT, CASE 3)
                    if address_with_ftpt.loc[non_adu_indices, 'FootprintID_right'].notna().all():
                                                
                        # Associate information with one point per footprint, making notes for all other points 
                        address_merge, drop_idx = inv_local.associate_scraped_split_multiple_addresses(address_merge, non_adu_indices, row, scraped_merge.columns, address_with_ftpt, 'FootprintID_right', use_height)
                        address_merge.loc[adu_index, 'NoScrapedData'] = 'Scraped Data Associated with Non-ADU Structure'                     

                    # If parcel contains ADU(s), and some or all non-ADU points are not located within a footprint (MANUSCRIPT CASE 4)
                    else:
                    
                        # Find the nearest footprint for each non-ADU address 
                        address_with_ftpt = inv_local.find_nearest(address_with_ftpt, parcel_ftpt)
                        non_adu = address_with_ftpt.loc[non_adu_indices]
                        address_with_ftpt_close_idx = non_adu[non_adu['DistanceToFtpt'] <= 10].index
                        address_with_ftpt_far_idx = non_adu[non_adu['DistanceToFtpt'] > 10].index

                        # If there are no address points close to footprints
                        if len(address_with_ftpt_close_idx) == 0: 

                            # Reset footprints and building area for cases where data is close and map appropriate information to address_with_ftpt
                            address_merge.loc[non_adu, 'FootprintID'] = address_with_ftpt.loc[non_adu, 'ClosestFtpt_ID'].values
                            address_with_ftpt = inv_local.map_closest_footprints(address_with_ftpt.copy(), footprints_orig, use_height)

                            # Associate information with one point per footprint, making notes for all other points 
                            address_merge.loc[non_adu, 'NoScrapedData'] = 'Has Scraped Data, but 10m Away - No Close Points' 
                            address_merge.loc[adu_index, 'NoScrapedData'] = 'Scraped Data Associated with Non-ADU Structure'  
                            address_merge, drop_idx = inv_local.associate_scraped_split_multiple_addresses(address_merge, non_adu, row, scraped_merge.columns, address_with_ftpt, 'ClosestFtpt_ID', use_height)

                        # If there are some address points close to footprints
                        else:    
                            # Reset footprints and building area for cases where data is close and map appropriate information to address_with_ftpt
                            address_merge.loc[address_with_ftpt_close_idx, 'FootprintID'] = address_with_ftpt.loc[address_with_ftpt_close_idx, 'ClosestFtpt_ID'].values
                            address_with_ftpt = inv_local.map_closest_footprints(address_with_ftpt.copy(), footprints_orig, use_height)

                            # Associate information with one point per footprint, making notes for all other points 
                            address_merge, drop_idx = inv_local.associate_scraped_split_multiple_addresses(address_merge, address_with_ftpt_close_idx, row, scraped_merge.columns, address_with_ftpt, 'ClosestFtpt_ID', use_height)
                            address_merge.loc[address_with_ftpt_far_idx, 'NoScrapedData'] = 'Point Not Within 10m of Footprint'  
                            address_merge.loc[adu_index, 'NoScrapedData'] = 'Scraped Data Associated with Non-ADU Structure' 
                    
                # Only ADU points present in the parcel
                else: 
                    # NOTE: This case only appears once in Hayward. More robust code may be needed if there are additional cases
                    address_merge = inv_local.associate_scraped_data(address_merge, gdf_index[0], row, scraped_merge.columns)
                    address_merge.loc[gdf_index[1:], 'NoScrapedData'] = 'Duplicate Address within Parcel / Footprint'  
                    address_merge.loc[gdf_index[0], 'FC_Updated'] = 'Single Family Residential - Converted from Only ADU'

            
    if index % 1000 == 0: 
        print(index, 'parcels assessed')
    
# Ensure all footprint values are numeric 
address_merge.loc[:,'FootprintID'] = pd.to_numeric(address_merge['FootprintID'], errors='coerce')

# Export footprint-level inventory
inv_local.gdf_to_json(address_merge, dir_intermediate + 'Address_Merged.json')